In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def numerical_gradient(f, params, h=1e-5):
#Central-difference numerical gradient of function f at params.
    params = np.asarray(params, dtype=float)
    grad = np.zeros_like(params)     #placeholder
    for i in range(len(params)):
        step = np.zeros_like(params)
        step[i] = h
        grad[i] = (f(params + step) - f(params - step)) / (2 * h)   #formula
    return grad

def gradient_descent(f, x0, lr=0.01, h=1e-5, tol=1e-10, max_iter=20000,
                      backtrack=True, shrink=0.5, max_backtrack=60,
                      normalize=True):
#Multi-dimensional gradient descent with numerical derivatives.
#normalize=True: step along the unit direction -grad/|grad| rather than -lr*grad directly. 
#This decouples the step size from how large the gradient happens to be 
#chi^2 can have a huge gradient far from the minimum, which would otherwise cause a
#single step (even after a few backtracks) to overshoot into an unphysical region. 
#still plain steepest descent, just with the distance moved per step controlled directly by `step`.

#backtrack=True: if the proposed step does not decrease f, 
#shrink the step size (repeatedly) until it does, or give up after max_backtrack tries.

#Returns: best-fit params, history of f(x) at every step.
    
    x = np.asarray(x0, dtype=float).copy()
    f_cur = f(x)
    history = [f_cur]
    step = lr
    for it in range(max_iter):
        grad = numerical_gradient(f, x, h=h)
        g_norm = np.linalg.norm(grad)
        if g_norm == 0 or not np.isfinite(g_norm):
            break
        direction = grad / g_norm if normalize else grad

        if backtrack:
            trial_step = step
            improved = False
            for _ in range(max_backtrack):
                x_new = x - trial_step * direction
                f_new = f(x_new)
                if np.isfinite(f_new) and f_new < f_cur:
                    improved = True
                    break
                trial_step *= shrink
            if not improved:
                break
            step = trial_step * 1.5    #try slightly larger step next time
        else:
            x_new = x - lr * direction
            f_new = f(x_new)

        history.append(f_new)
        if abs(f_cur - f_new) < tol:
            x, f_cur = x_new, f_new
            break
        x, f_cur = x_new, f_new

    return x, np.array(history)

#Testing:

def f_test(p):
    x, y = p
    return (x - 2)**2 + (y - 2)**2

x0 = np.array([-3.0, 5.0])     #deliberately far from the minimum (2,2)
best, hist = gradient_descent(f_test, x0, lr=0.1, max_iter=1000)

print("Starting point:", x0)
print("Best-fit (x,y):", best)
print("f at minimum:", hist[-1])
print("Number of steps taken:", len(hist)-1)

#Plot: convergence of f vs iteration number
fig, ax = plt.subplots(figsize=(6,4.5))
ax.plot(hist, lw=1.8)
ax.set_yscale("log")
ax.set_xlabel("step i")
ax.set_ylabel(r"$f(x,y)$")
ax.set_title("Gradient descent test: $f(x,y)=(x-2)^2+(y-2)^2$")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("toy_function_test.png", dpi=150)
print("saved plot")


#load data
data = np.loadtxt("/Users/nichaeadnan/Downloads/smf_cosmos.dat")
logM_obs = data[:,0]
n_obs    = data[:,1]
sigma    = data[:,2]
M_obs    = 10**logM_obs   #actual mass (not log) for plugging into the Schechter formula

def schechter(Mgal, phi_star, M_star, alpha):
    x = Mgal / M_star
    return phi_star * x**(alpha+1) * np.exp(-x) * np.log(10)

def chi2_from_logparams(p):
#p = [log10(phi_star), log10(M_star), alpha]
    log_phi, log_Mstar, alpha = p
    phi_star = 10**log_phi
    M_star   = 10**log_Mstar
    model = schechter(M_obs, phi_star, M_star, alpha)
    return np.sum(((n_obs - model)/sigma)**2)

#three starting points, to test robustness
starts = {
    "start A": np.array([np.log10(1e-3), np.log10(10**10.5), -1.0]),
    "start B": np.array([np.log10(1e-2), np.log10(10**11.0), -1.5]),
    "start C": np.array([np.log10(1e-4), np.log10(10**10.0), -0.5]),
}

results = {}
for name, p0 in starts.items():
    best, hist = gradient_descent(chi2_from_logparams, p0, lr=5e-4, h=1e-6, max_iter=30000)
    phi_star = 10**best[0]; M_star = 10**best[1]; alpha = best[2]
    results[name] = dict(p0=p0, best=best, hist=hist,
                          phi_star=phi_star, M_star=M_star, alpha=alpha)
    print(f"{name}: start logM*={p0[1]:.2f}, alpha0={p0[2]:.2f}  ->  "
          f"phi*={phi_star:.4e}, M*={M_star:.4e}, alpha={alpha:.4f}, "
          f"chi2={hist[-1]:.4f}, nsteps={len(hist)-1}")

#Plot (a): chi^2 vs step, for all three starts
fig, ax = plt.subplots(figsize=(6.5,4.5))
colors = {"start A":"C0", "start B":"C1", "start C":"C2"}
for name, r in results.items():
    ax.plot(r["hist"], label=f'{name} (final χ²={r["hist"][-1]:.3f})',
            color=colors[name], lw=1.6)
ax.set_yscale("log")
ax.set_xlabel("step i")
ax.set_ylabel(r"$\chi^2$")
ax.set_title("Schechter fit: convergence from 3 different starting points")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("schechter_chi2_vs_step.png", dpi=150)

#Plot (b): best-fit model vs data
best = results["start A"]["best"]   # all three starts agree, so pick one
phi_star = 10**best[0]; M_star = 10**best[1]; alpha = best[2]

Mgal_grid = np.logspace(9.4, 12.0, 400)
n_model_grid = schechter(Mgal_grid, phi_star, M_star, alpha)

fig, ax = plt.subplots(figsize=(6.5,5))
ax.errorbar(M_obs, n_obs, yerr=sigma, fmt='o', ms=5, color='k',
            label='COSMOS data', capsize=2, zorder=3)
ax.plot(Mgal_grid, n_model_grid, '-', color='crimson', lw=2,
        label=(r'best-fit Schechter'+'\n'
               rf'$\phi_*={phi_star:.3e}$'+'\n'
               rf'$M_*={M_star:.3e}\,M_\odot$'+'\n'
               rf'$\alpha={alpha:.3f}$'))
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$M_{gal}$ [$M_\odot$]')
ax.set_ylabel(r'$n(M_{gal})$ [dex$^{-1}$ (Mpc/h)$^{-3}$]')
ax.set_title('Best-fit Schechter function vs. COSMOS stellar mass function')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')
fig.tight_layout()
fig.savefig("schechter_bestfit_vs_data.png", dpi=150)

print()
print("FINAL BEST FIT (all three starts agree):")
print(f"  phi_star = {phi_star:.4e}  [dex^-1 (Mpc/h)^-3]")
print(f"  M_star   = {M_star:.4e}  [Msun]   (log10 M_star = {best[1]:.4f})")
print(f"  alpha    = {alpha:.4f}")
print(f"  chi2     = {results['start A']['hist'][-1]:.4f}  (dof = {len(n_obs)-3})")


Starting point: [-3.  5.]
Best-fit (x,y): [1.99999369 2.00000379]
f at minimum: 5.413753216020696e-11
Number of steps taken: 24
saved plot
start A: start logM*=10.50, alpha0=-1.00  ->  phi*=2.7025e-03, M*=9.4676e+10, alpha=-1.0094, chi2=2.9081, nsteps=153
start B: start logM*=11.00, alpha0=-1.50  ->  phi*=2.7028e-03, M*=9.4669e+10, alpha=-1.0094, chi2=2.9081, nsteps=193
start C: start logM*=10.00, alpha0=-0.50  ->  phi*=2.7027e-03, M*=9.4672e+10, alpha=-1.0094, chi2=2.9081, nsteps=249

FINAL BEST FIT (all three starts agree):
  phi_star = 2.7025e-03  [dex^-1 (Mpc/h)^-3]
  M_star   = 9.4676e+10  [Msun]   (log10 M_star = 10.9762)
  alpha    = -1.0094
  chi2     = 2.9081  (dof = 9)
